In [1]:
# config

TICKERS = {
    "AAPL": 0.30,
    "MSFT": 0.30,
    "NVDA": 0.20,
    "AMZN": 0.20
}

BASE_VALUE = 100

In [2]:
#Engine
import redis
from config import TICKERS, BASE_VALUE

r = redis.Redis(host="localhost", port=6379)

base_prices = {}
current_prices = {}

def initialize_base_prices(prices):
    global base_prices
    base_prices = prices

def update_price(symbol, price):
    current_prices[symbol] = price

def compute_index():
    index = 0
    for symbol, weight in TICKERS.items():
        if symbol not in base_prices:
            return None
        price_rel = current_prices[symbol] / base_prices[symbol]
        index += weight * price_rel

    index *= BASE_VALUE

    r.set("index_value", index)

    return index

ImportError: cannot import name 'TICKERS' from 'config' (c:\Users\praxy\anaconda3\Lib\site-packages\config\__init__.py)

In [ ]:
#Real time stream
import websocket
import json
from index_calculator import update_price, compute_index

API_KEY = "YOUR_KEY"

def on_message(ws, message):
    data = json.loads(message)
    for trade in data:
        symbol = trade.get("sym")
        price = trade.get("p")
        if symbol:
            update_price(symbol, price)
            index = compute_index()
            if index:
                print("Index:", index)


def on_open(ws):

    auth = {"action": "auth", "params": API_KEY}
    ws.send(json.dumps(auth))
    subs = "T.AAPL,T.MSFT,T.NVDA,T.AMZN"
    ws.send(json.dumps({
        "action": "subscribe",
        "params": subs
    }))


socket = websocket.WebSocketApp(
    "wss://socket.polygon.io/stocks",
    on_open=on_open,
    on_message=on_message
)

socket.run_forever()

In [ ]:
#Creating the API
from fastapi import FastAPI
import redis

app = FastAPI()
r = redis.Redis(host="localhost", port=6379)
@app.get("/index")

def get_index():
    value = r.get("index_value")
    return {
        "index": "MY_CUSTOM_INDEX",
        "value": float(value)
    }

In [ ]:
#commands for the terminal 
# uvicorn api.main:app --reload
#Example of endponit 
#http://localhost:8000/index

In [ ]:
#It should have the response
{
 "index": "MY_CUSTOM_INDEX",
 "value": 104.23
}